# EDA for thesis

Graph-based optimizations of public transit routes

Data source: [GTFS realtime vehicle positions, Lviv Open Data](https://opendata.city-adm.lviv.ua/dataset/lviv-public-transport-gtfs-real-time/resource/d45fd95a-ffc9-45b1-be05-52012707d51f?inner_span=True)

[GTFS columns reference](https://gtfs.org/documentation/schedule/reference/)

## Installing dependencies

In [16]:
%pip install gtfs_kit osmnx numpy polars contextily folium mapclassify networkx matplotlib seaborn --quiet
# %pip install --pre geopolars

Note: you may need to restart the kernel to use updated packages.


## Constants

In [17]:
from pathlib import Path


FOLDER_PATH = Path("../static")
DELIMITER = '-' * 20
ALPHA_PREFIX_REGEX = r"^\D+"

GTFS_ROUTE_TYPES = {
    "0": "Tram, Streetcar, Light rail",
    "1": "Subway, Metro",
    "2": "Rail",
    "3": "Bus",
    "4": "Ferry",

    # street-level rail cars where the cable runs beneath the vehicle
    "5": "Cable tram",

    # e.g., gondola lift, aerial tramway
    "6": "Aerial lift, suspended cable car",

    "7": "Funicular",
    "11": "Trolleybus",
    "12": "Monorail",
}

def partial_gtfs_route_types(route_types: list[str]) -> dict[str, str]:
    """
    Returns a dictionary of GTFS route types for the given list of route types.
    """
    return {k: v for k, v in GTFS_ROUTE_TYPES.items() if k in route_types}

## GTFS static data analysis

In [18]:
import polars as pl


print("Static GTFS info directory:", FOLDER_PATH.resolve(), end='\n\n')
dataframes = {
    f.name.split(".")[0]: pl.read_csv(f.resolve(), infer_schema_length=None)
    for f in FOLDER_PATH.iterdir() if f.is_file()
}

with pl.Config(tbl_formatting="MARKDOWN") as cfg:
    cfg.set_tbl_cols(-1)
    cfg.set_tbl_rows(-1)

    for name, df in dataframes.items():
        print(DELIMITER, name, DELIMITER, end='\n\n')

        print('Sample')
        print(df.head(3), end='\n\n')

Static GTFS info directory: /home/rojikaru/Projects/gtfs-tryout/static

-------------------- agency --------------------

Sample
shape: (3, 8)
| agency_id | agency_nam | agency_url | agency_tim | agency_la | agency_ph | agency_fa | agency_em |
| ---       | e          | ---        | ezone      | ng        | one       | re_url    | ail       |
| i64       | ---        | str        | ---        | ---       | ---       | ---       | ---       |
|           | str        |            | str        | str       | str       | str       | str       |
|-----------|------------|------------|------------|-----------|-----------|-----------|-----------|
| 31        | Міра і К   | http://cit | Europe/Kie | uk        | null      | null      | null      |
|           |            | y-adm.lviv | v          |           |           |           |           |
|           |            | .ua/portal |            |           |           |           |           |
|           |            | …          |          

### Trips

In [19]:
trips_df = dataframes["trips"]

print(DELIMITER, "Wheelchair accessible scheduled vehicles", DELIMITER)
print(trips_df["wheelchair_accessible"].value_counts())

-------------------- Wheelchair accessible scheduled vehicles --------------------
shape: (2, 2)
┌───────────────────────┬───────┐
│ wheelchair_accessible ┆ count │
│ ---                   ┆ ---   │
│ i64                   ┆ u32   │
╞═══════════════════════╪═══════╡
│ 1                     ┆ 2203  │
│ 0                     ┆ 13711 │
└───────────────────────┴───────┘


## Routes

In [20]:
routes_df = dataframes["routes"]
type_col, name_col, prefix_col = (
    "route_type",
    "route_short_name",
    "route_short_name_prefix",
)
pl_type_col, pl_name_col, pl_prefix_col = (
    pl.col(type_col),
    pl.col(name_col),
    pl.col(prefix_col),
)

print("Total routes:", routes_df.shape[0], end="\n\n")

# route_type is assigned incorrectly for trolleybuses, should be 11
# https://gtfs.org/documentation/schedule/reference/#routestxt
print(DELIMITER, "Route types (original)", DELIMITER)
print(routes_df[type_col].value_counts().sort("count"), end="\n\n")

prefixes_df = routes_df.with_columns(
    pl_name_col.str.extract(ALPHA_PREFIX_REGEX, 0).alias(prefix_col)
)

print(DELIMITER, "Route types (inferred)", DELIMITER)
print(prefixes_df[prefix_col].value_counts().sort("count"), end="\n\n")

trolley_prefix, tram_prefix = "Тр", "Т"

assert (
    prefixes_df[prefix_col].null_count() == 0
), f"Regex parsing is broken (new route without an alpha prefix in {name_col}?)"
assert (
    prefixes_df.filter((pl_type_col == 0) & (pl_prefix_col == trolley_prefix)).shape[0]
    == 0
), "Trolleys aren't distinguished from trams"
assert (
    prefixes_df.filter((pl_type_col == 11) & (pl_prefix_col == tram_prefix)).shape[0]
    == 0
), "Trams aren't distinguished from trolleys"

prefixes_df = prefixes_df.with_columns(
    pl.when(pl_prefix_col == trolley_prefix)
    .then(11)
    .otherwise(pl_type_col)
    .alias(type_col)
)

routes_df = prefixes_df.drop(prefix_col)

print(DELIMITER, "Route types (corrected)", DELIMITER)
print(routes_df[type_col].value_counts().sort("count"))

Total routes: 72

-------------------- Route types (original) --------------------
shape: (2, 2)
┌────────────┬───────┐
│ route_type ┆ count │
│ ---        ┆ ---   │
│ i64        ┆ u32   │
╞════════════╪═══════╡
│ 0          ┆ 8     │
│ 3          ┆ 64    │
└────────────┴───────┘

-------------------- Route types (inferred) --------------------
shape: (3, 2)
┌─────────────────────────┬───────┐
│ route_short_name_prefix ┆ count │
│ ---                     ┆ ---   │
│ str                     ┆ u32   │
╞═════════════════════════╪═══════╡
│ Т                       ┆ 8     │
│ Тр                      ┆ 9     │
│ А                       ┆ 55    │
└─────────────────────────┴───────┘

-------------------- Route types (corrected) --------------------
shape: (3, 2)
┌────────────┬───────┐
│ route_type ┆ count │
│ ---        ┆ ---   │
│ i64        ┆ u32   │
╞════════════╪═══════╡
│ 0          ┆ 8     │
│ 11         ┆ 9     │
│ 3          ┆ 55    │
└────────────┴───────┘


In [ ]:
agency_df = dataframes["agency"]

routes_with_agency_df = routes_df.join(agency_df, on="agency_id", how="left")
assert (
    
    routes_df.shape[0] == routes_with_agency_df.shape[0]
), "LEFT JOIN with agency_df changed the number of rows, something is wrong"

crosstab_df = (
    routes_with_agency_df.pivot(
        values=type_col,
        index=["agency_id", "agency_name"],
        on=type_col,
        aggregate_function="len",
    )
).rename(partial_gtfs_route_types(["0", "3", "11"]))

print(DELIMITER, "Route types by agency", DELIMITER)
print(crosstab_df, end="\n\n")

-------------------- Route types by agency --------------------
shape: (7, 5)
┌───────────┬───────────────────────┬─────┬─────────────────────────────┬────────────┐
│ agency_id ┆ agency_name           ┆ Bus ┆ Tram, Streetcar, Light rail ┆ Trolleybus │
│ ---       ┆ ---                   ┆ --- ┆ ---                         ┆ ---        │
│ i64       ┆ str                   ┆ u32 ┆ u32                         ┆ u32        │
╞═══════════╪═══════════════════════╪═════╪═════════════════════════════╪════════════╡
│ 31        ┆ Міра і К              ┆ 6   ┆ 0                           ┆ 0          │
│ 52        ┆ АТП-1                 ┆ 28  ┆ 0                           ┆ 0          │
│ 89        ┆ ЛКП Львівелектротранс ┆ 0   ┆ 8                           ┆ 9          │
│ 148       ┆ Фіакр-Львів           ┆ 6   ┆ 0                           ┆ 0          │
│ 32        ┆ Львівське АТП-14630   ┆ 6   ┆ 0                           ┆ 0          │
│ 327       ┆ ТОВ Епітранс          ┆ 5   ┆ 0       

In [31]:
with pl.Config(tbl_formatting="MARKDOWN") as cfg:
    cfg.set_tbl_cols(-1)
    cfg.set_tbl_rows(-1)
    print(DELIMITER, "Route types by agency", DELIMITER)
    routes_with_agency_filter_df = routes_with_agency_df.select(
        ["agency_id", "agency_name", "route_short_name"]
    ).filter(pl.col("agency_name") == "Успіх БМ")
    print(routes_with_agency_filter_df, end="\n\n")

-------------------- Route types by agency --------------------
shape: (4, 3)
| agency_id | agency_name | route_short_name |
| ---       | ---         | ---              |
| i64       | str         | str              |
|-----------|-------------|------------------|
| 10        | Успіх БМ    | А84              |
| 10        | Успіх БМ    | А62              |
| 10        | Успіх БМ    | А51              |
| 10        | Успіх БМ    | А53              |



In [25]:
trips_with_routes_df = trips_df.join(routes_with_agency_df, on="route_id", how="left")
assert (
    trips_df.shape[0] == trips_with_routes_df.shape[0]
), "LEFT JOIN with routes_with_agency_df changed the number of rows, something is wrong"

print(DELIMITER, "Wheelchair accessible scheduled vehicles by agency", DELIMITER)
print(
    trips_with_routes_df.group_by(["agency_id", "agency_name"])
    .agg(
        [
            pl.mean("wheelchair_accessible").alias("wheelchair_accessible_share"),
            pl.count("trip_id").alias("total_trips"),
        ]
    )
    .sort("wheelchair_accessible_share", descending=True)
)

-------------------- Wheelchair accessible scheduled vehicles by agency --------------------
shape: (7, 4)
┌───────────┬───────────────────────┬─────────────────────────────┬─────────────┐
│ agency_id ┆ agency_name           ┆ wheelchair_accessible_share ┆ total_trips │
│ ---       ┆ ---                   ┆ ---                         ┆ ---         │
│ i64       ┆ str                   ┆ f64                         ┆ u32         │
╞═══════════╪═══════════════════════╪═════════════════════════════╪═════════════╡
│ 10        ┆ Успіх БМ              ┆ 0.637509                    ┆ 1349        │
│ 148       ┆ Фіакр-Львів           ┆ 0.187441                    ┆ 1051        │
│ 89        ┆ ЛКП Львівелектротранс ┆ 0.167337                    ┆ 4972        │
│ 52        ┆ АТП-1                 ┆ 0.056699                    ┆ 5538        │
│ 327       ┆ ТОВ Епітранс          ┆ 0.0                         ┆ 798         │
│ 32        ┆ Львівське АТП-14630   ┆ 0.0                         ┆ 996  

In [34]:
stop_times_df = dataframes["stop_times"]
stop_counts_df = (
    stop_times_df.group_by("trip_id")
    .agg(pl.count("stop_id").alias("stop_count"))
    .sort("stop_count", descending=True)
)
stop_counts_with_routes_df = stop_counts_df.join(
    trips_with_routes_df, on="trip_id", how="left"
    # TODO: change _name to _id
).select(["trip_id", "route_short_name", "agency_name", "stop_count"])
assert (
    stop_counts_df.shape[0] == stop_counts_with_routes_df.shape[0]
), "LEFT JOIN with trips_with_routes_df changed the number of rows, something is wrong"

stop_counts_with_routes_df = (
    # TODO: change _name to _id
    stop_counts_with_routes_df.group_by(["agency_name", "route_short_name"])
    .agg(
        [
            pl.mean("stop_count").alias("mean_stop_count"),
            pl.median("stop_count").alias("median_stop_count"),
            pl.count("trip_id").alias("total_trips"),
        ]
    )
    .sort("mean_stop_count", descending=True)
)

print(DELIMITER, "Stop counts per trip", DELIMITER)
with pl.Config(tbl_formatting="MARKDOWN") as cfg:
    cfg.set_tbl_cols(-1)
    cfg.set_tbl_rows(-1)
    print(stop_counts_with_routes_df, end="\n\n")

-------------------- Stop counts per trip --------------------
shape: (72, 5)
| agency_name           | route_short_name | mean_stop_count | median_stop_count | total_trips |
| ---                   | ---              | ---             | ---               | ---         |
| str                   | str              | f64             | f64               | u32         |
|-----------------------|------------------|-----------------|-------------------|-------------|
| АТП-1                 | А47              | 52.015251       | 53.0              | 459         |
| Львівське АТП-14630   | А23              | 48.099206       | 46.0              | 252         |
| Міра і К              | А25              | 47.5            | 47.5              | 232         |
| АТП-1                 | А09              | 45.495549       | 44.0              | 337         |
| Фіакр-Львів           | А41              | 43.329949       | 42.0              | 197         |
| АТП-1                 | А61              | 42.8